## Lab 6: Decision Trees

In this lab you'll build and tune a decision tree to predict **forest cover type** from wilderness cartographic data. This is multi-class classification problem used by the US Forest Service to map land cover without costly field surveys.

**Dataset:** UCI Forest Cover Type (id=31). 581,012 records from four wilderness areas in the Roosevelt National Forest, Colorado. Features include elevation, slope, hillshade, distance to water and roads, wilderness area, and soil type. Our response varaible, `Cover_Type`, includes 7 different forest types: Spruce/Fir, Lodgepole Pine, Ponderosa Pine, Cottonwood/Willow, Aspen, Douglas-fir, Krummholz (encoded as integers from 1 to 7). More information on the data can be found [here](https://archive.ics.uci.edu/dataset/31/covertype).


### Step 1: Load and Explore the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
forest = fetch_ucirepo(id=31)

df = pd.concat([forest.data.features, forest.data.targets], axis=1)

print(f"Full dataset shape: {df.shape}")
print()
print("Cover type distribution (full dataset):")
print(df['Cover_Type'].value_counts().sort_index())
df.head()

In [ ]:
df.info()

### Step 2: Preprocess

The Forest Cover Type dataset is already clean and fully numeric, meaning all features are encoded as integers. The 54 features include:

- **10 continuous:** elevation, aspect, slope, hillshade indices, distances to hydrology, roadways, and fire points
- **44 binary:** 4 wilderness area indicators and 40 soil type indicators

No encoding or scaling is needed. We sample 30,000 rows to keep lab runtimes manageable.

In [65]:
df_sample = df.sample(n=30_000, random_state=42)

class_names = ['Spruce/Fir', 'Lodgepole Pine', 'Ponderosa Pine',
               'Cottonwood/Willow', 'Aspen', 'Douglas-fir', 'Krummholz']

features = [c for c in df_sample.columns if c != 'Cover_Type']
X = df_sample[features]
y = df_sample['Cover_Type']

print("Sampled class distribution:")
print(y.value_counts().sort_index())

Sampled class distribution:
Cover_Type
1    11064
2    14558
3     1830
4      137
5      473
6      901
7     1037
Name: count, dtype: int64


In [66]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print(f"Training samples: {len(X_train):,}  |  Test samples: {len(X_test):,}")

Training samples: 20,100  |  Test samples: 9,900


### Step 3: Fit a Decision Tree

In [67]:
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

train_acc = accuracy_score(y_train, dt_full.predict(X_train))
test_acc  = accuracy_score(y_test,  dt_full.predict(X_test))

print(f"Unconstrained tree depth:  {dt_full.get_depth()}")
print(f"Training accuracy:         {train_acc:.3f}")
print(f"Test accuracy:             {test_acc:.3f}")

Unconstrained tree depth:  35
Training accuracy:         1.000
Test accuracy:             0.770


##### Uh oh! Look at the difference between our test accuracy and our training accuracy. It looks like our model is overfitting! Let's try tuning to prevent this. 

### Step 4: Tune `max_depth` with Cross-Validation

Rather than evaluating each depth on the held-out test set (which leaks information), we use **5-fold cross-validation** on the training data. The shaded band shows ±1 standard deviation across folds.

In [ ]:
depths = range(1, 21)
cv_means, cv_stds = [], []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds  = np.array(cv_stds)
best_depth = list(depths)[np.argmax(cv_means)]
print(f"Best max_depth (5-fold CV): {best_depth}  (CV accuracy: {cv_means.max():.3f})")

plt.figure(figsize=(9, 4))
plt.plot(list(depths), cv_means, label='CV Mean Accuracy', marker='o')
plt.fill_between(list(depths), cv_means - cv_stds, cv_means + cv_stds,
                 alpha=0.2, label=r'$\pm$1 SD')
plt.axvline(best_depth, color='gray', linestyle='--', label=f'Best depth = {best_depth}')
plt.xlabel('max_depth')
plt.ylabel('CV Accuracy')
plt.title('Decision Tree: 5-Fold CV Accuracy vs. Tree Depth')
plt.legend()
plt.tight_layout()
plt.show()

**Discussion Questions**

1. At what depth does CV accuracy stop improving meaningfully? What does this suggest about how complex the underlying patterns in the data are?
2. Why does the uncertainty band (±1 SD) tend to widen as depth increases?
3. Why is it important to use cross-validation here rather than just checking accuracy on the test set for each depth?

### Step 5: Tune `min_samples_leaf` with Cross-Validation

`min_samples_leaf` sets the minimum number of training samples required at any leaf node. Larger values smooth the tree and act as a second form of regularization alongside `max_depth`.

In [ ]:
leaf_sizes = [1, 5, 10, 25, 50, 100]
leaf_cv_means, leaf_cv_stds = [], []

for n in leaf_sizes:
    dt = DecisionTreeClassifier(max_depth=best_depth, min_samples_leaf=n, random_state=42)
    scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
    leaf_cv_means.append(scores.mean())
    leaf_cv_stds.append(scores.std())

leaf_cv_means = np.array(leaf_cv_means)
leaf_cv_stds  = np.array(leaf_cv_stds)
best_leaf = leaf_sizes[np.argmax(leaf_cv_means)]
print(f"Best min_samples_leaf (5-fold CV): {best_leaf}  (CV accuracy: {leaf_cv_means.max():.3f})")

plt.figure(figsize=(9, 4))
plt.plot(leaf_sizes, leaf_cv_means, label='CV Mean Accuracy', marker='o', color='steelblue')
plt.fill_between(leaf_sizes, leaf_cv_means - leaf_cv_stds, leaf_cv_means + leaf_cv_stds,
                 alpha=0.2, color='steelblue')
plt.axvline(best_leaf, color='gray', linestyle='--', label=f'Best min_samples_leaf = {best_leaf}')
plt.xlabel('min_samples_leaf')
plt.ylabel('CV Accuracy')
plt.title(f'Decision Tree: 5-Fold CV Accuracy vs. min_samples_leaf (max_depth={best_depth})')
plt.legend()
plt.tight_layout()
plt.show()

**Discussion Questions**

1. How sensitive is CV accuracy to changes in `min_samples_leaf`? Does performance degrade sharply or gradually as leaf size increases?
2. `max_depth` and `min_samples_leaf` both prevent the tree from being too specific, but they do so differently. In your own words, how does each one constrain the tree?

### Step 6: Joint Tuning with `GridSearchCV`

Tuning hyperparameters one at a time ignores their interactions. `GridSearchCV` evaluates every combination in a parameter grid using cross-validation, finding the jointly optimal settings.

In [ ]:
param_grid = {
    'max_depth':        [5, 10, 15, 20],
    'min_samples_leaf': [1, 5, 10, 25],
    'criterion':        ['gini', 'entropy']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Best parameters:  {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")

dt_best = grid_search.best_estimator_
dt_acc  = accuracy_score(y_test, dt_best.predict(X_test))
print(f"Test accuracy:    {dt_acc:.3f}")

**Discussion Questions**

1. Does the best `max_depth` from GridSearchCV match what you found by tuning it individually in Step 4? If not, why might joint tuning arrive at a different answer?
2. What criterion (gini or entropy) did GridSearchCV select? In practice, these two measures of impurity often give similar results — did you see a meaningful difference here?
3. How close is the best CV accuracy to the final test accuracy? A large gap would suggest the model is still overfitting; a small gap suggests it generalizes well.

### Step 7: Visualize the Best Tree

With 7 cover types and many features, even a shallow tree captures meaningful ecological patterns. Elevation tends to dominate early splits — higher elevations favor Spruce/Fir and Krummholz, lower elevations favor Ponderosa Pine.

In [63]:
dt_viz = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=grid_search.best_params_['min_samples_leaf'],
    criterion=grid_search.best_params_['criterion'],
    random_state=42
)
dt_viz.fit(X_train, y_train)

plt.figure(figsize=(22, 10))
plot_tree(dt_viz, feature_names=features, class_names=class_names,
          filled=True, rounded=True, fontsize=7)
plt.title('Decision Tree — Forest Cover Type (depth=3 for display)')
plt.tight_layout()
plt.show()

### Step 9: Evaluate with Classification Report

Overall accuracy averages across all 7 classes equally. Since some cover types are much rarer than others (Cottonwood/Willow is <1% of the sample), the per-class report reveals where the model struggles.

In [48]:
y_pred = dt_best.predict(X_test)
print(classification_report(y_test, y_pred, target_names=class_names))

                   precision    recall  f1-score   support

       Spruce/Fir       0.77      0.75      0.76      3641
   Lodgepole Pine       0.80      0.81      0.80      4809
   Ponderosa Pine       0.73      0.79      0.75       573
Cottonwood/Willow       0.82      0.60      0.69        47
            Aspen       0.46      0.44      0.45       176
      Douglas-fir       0.58      0.53      0.55       299
        Krummholz       0.75      0.75      0.75       355

         accuracy                           0.77      9900
        macro avg       0.70      0.66      0.68      9900
     weighted avg       0.77      0.77      0.77      9900



**Discussion Questions**

1. What feature appears at the root node (the very first split)? Does it make ecological sense that this feature is the most informative for separating forest cover types?
2. Look at the elevation threshold used at the root split (in meters). Rocky Mountain subalpine forests typically begin around 2,750–3,000m. Does the split value align with a meaningful ecological boundary?
3. Follow the left branch (lower elevation) to its leaf node. What cover type does the tree predict there? Follow the right branch (higher elevation). Do the predicted cover types match what you would expect ecologically?
4. Each node shows `value = [n1, n2, n3, n4, n5, n6, n7]` — the count of training samples from each of the 7 cover types in that region. Look at the leaf nodes: which are relatively pure (one class dominates) and which are still mixed across several types?
5. The node color represents the majority class. Are there any nodes that appear pale or washed out? What does a pale node indicate about confidence in that prediction?
6. Cottonwood/Willow (class 4) is extremely rare — under 0.5% of the full dataset. Can you find it as a majority class in any leaf node of this depth-3 tree? What does its absence suggest about how decision trees handle rare classes?

### Step 8: Feature Importance

With 54 features, many soil type indicators will have near-zero importance. The plot below shows the top 15 most important features.

In [ ]:
feat_imp = pd.Series(dt_best.feature_importances_, index=features)\
             .sort_values(ascending=False)

plt.figure(figsize=(7, 5))
feat_imp.head(15).sort_values().plot(kind='barh', color='steelblue')
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.tight_layout()
plt.show()

**Discussion Questions**

1. Which feature is most important? Is this the same feature that appeared at the root node of the tree in Step 7?
2. The dataset includes 40 binary soil type features (Soil_Type1–Soil_Type40). How many appear in the top 15? What does this suggest about the relative predictive power of soil type versus topographic features like elevation and slope?
3. Why might `Elevation` carry more predictive power than `Aspect` or `Slope`, even though all three describe the terrain?

### Step 9: Evaluate with Classification Report

Overall accuracy averages across all 7 classes equally. Since some cover types are much rarer than others (Cottonwood/Willow is <1% of the sample), the per-class report reveals where the model struggles.

In [ ]:
y_pred = dt_best.predict(X_test)
print(classification_report(y_test, y_pred, target_names=class_names))

**Discussion Questions**

1. Which cover type has the lowest F1-score? Look at its support value (number of test samples). Is poor performance related to how rarely it appears in the data?
2. Compare the recall for Cottonwood/Willow versus Lodgepole Pine. What does a low recall mean in practice — what kind of errors is the model making for that class?
3. Overall accuracy is a single number, but this report shows very different performance across classes. In a real forest management application, why might overall accuracy be a misleading metric to report alone?
4. If you were a forest manager trying to identify old-growth Spruce/Fir habitat for conservation, would you care more about precision or recall for that class? Why?